# Demon Attack: Pixel States to VQ-VAE Discrete IDs

A deliberately small, visual smoke test for the Atari state discretizer used by `dqn_atari_vqvae_masked.py`. It collects a small random-policy dataset, briefly trains the same convolutional VQ-VAE, and shows the discrete ID assigned to each four-frame state stack.

> This validates that the encoder and quantizer run end-to-end. A short run is **not** evidence that the learned IDs have good semantic meaning; use the full experiment for that.

## 1. Imports and small-run settings

In [8]:
from collections import defaultdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Allow the notebook to run from either the repository root or src/.
repo_src = Path.cwd() / 'src'
if repo_src.exists():
    sys.path.insert(0, str(repo_src))

from dqn_atari_vqvae_masked import AtariVQVAE, make_env

SEED = 7
ENV_ID = 'ALE/DemonAttack-v5'
SAMPLE_SIZE = 256       # Keep this notebook quick and small.
EPOCHS = 5
BATCH_SIZE = 32
NUM_EMBEDDINGS = 32     # Smaller codebook makes this diagnostic easy to inspect.
EMBEDDING_DIM = 8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'Using {DEVICE}')

Using cpu


## 2. Collect a small Demon Attack state-transition sample

Each observation is the same preprocessed input used by training: four grayscale 84×84 frames.

In [10]:
env = make_env(ENV_ID, SEED)
action_names = env.unwrapped.get_action_meanings()
print('Observation shape:', env.observation_space.shape)
print('Full action space:', list(enumerate(action_names)))

states, next_states, actions = [], [], []
obs, _ = env.reset(seed=SEED)
for _ in range(SAMPLE_SIZE):
    action = env.action_space.sample()
    next_obs, _, terminated, truncated, _ = env.step(action)
    states.append(np.asarray(obs))
    next_states.append(np.asarray(next_obs))
    actions.append(action)
    obs = next_obs
    if terminated or truncated:
        obs, _ = env.reset()

states = np.asarray(states, dtype=np.uint8)
next_states = np.asarray(next_states, dtype=np.uint8)
actions = np.asarray(actions)
print('Collected:', states.shape)

NamespaceNotFound: Namespace ALE not found. Have you installed the proper package for ALE?

## 3. Train the same convolutional VQ-VAE briefly

In [ ]:
vqvae = AtariVQVAE(embedding_dim=EMBEDDING_DIM, num_embeddings=NUM_EMBEDDINGS).to(DEVICE)
optimizer = torch.optim.Adam(vqvae.parameters(), lr=1e-3)
loader = DataLoader(torch.from_numpy(states), batch_size=BATCH_SIZE, shuffle=True)
losses = []

vqvae.train()
for epoch in range(EPOCHS):
    epoch_losses = []
    for batch in loader:
        batch = batch.to(DEVICE).float() / 255.0
        reconstruction, vq_loss, _ = vqvae(batch)
        loss = F.mse_loss(reconstruction, batch) + vq_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    losses.append(np.mean(epoch_losses))
    print(f'Epoch {epoch + 1}/{EPOCHS}: loss={losses[-1]:.5f}')

plt.plot(losses, marker='o')
plt.title('Small-sample VQ-VAE training loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(alpha=0.3)
plt.show()

## 4. Assign and inspect discrete IDs

The code below obtains an ID for every current and next state. The image shown is the most recent frame in the four-frame stack.

In [ ]:
vqvae.eval()
with torch.no_grad():
    state_ids = vqvae.codes(torch.from_numpy(states).to(DEVICE).float() / 255.0).cpu().numpy()
    next_state_ids = vqvae.codes(torch.from_numpy(next_states).to(DEVICE).float() / 255.0).cpu().numpy()

unique_ids, counts = np.unique(state_ids, return_counts=True)
print('IDs used:', len(unique_ids), 'of', NUM_EMBEDDINGS)
print('Usage:', dict(zip(unique_ids.tolist(), counts.tolist())))

fig, axes = plt.subplots(4, 4, figsize=(12, 11))
for index, axis in enumerate(axes.flat):
    axis.imshow(states[index, -1], cmap='gray', vmin=0, vmax=255)
    axis.set_title(f'ID {state_ids[index]} → {next_state_ids[index]}\n{action_names[actions[index]]}', fontsize=9)
    axis.axis('off')
plt.suptitle('Sampled Demon Attack states and VQ-VAE discrete transitions', y=0.92)
plt.tight_layout()
plt.show()

## 5. Inspect action aliases for identical discrete transitions

If multiple actions appear for the same `(current_ID, next_ID)` pair, that is exactly the transition-alias condition used by the masking experiment.

In [ ]:
transition_actions = defaultdict(set)
for current_id, next_id, action in zip(state_ids, next_state_ids, actions):
    transition_actions[(int(current_id), int(next_id))].add(action_names[int(action)])

aliases = [(transition, sorted(action_set)) for transition, action_set in transition_actions.items() if len(action_set) > 1]
aliases.sort(key=lambda item: (-len(item[1]), item[0]))
print(f'Discrete transitions with more than one action: {len(aliases)}')
for transition, action_set in aliases[:20]:
    print(f'{transition[0]} → {transition[1]}: {action_set}')

# A self-loop is separately penalized by the training script.
print('Self-loop rate:', np.mean(state_ids == next_state_ids))
env.close()